# Module 2 - Detecting Epistemic Enclaves

**Time:** 11:15-12:45  
**Tool focus:** CDlib

This notebook compares several community-detection algorithms and then evaluates the resulting clusters as potential epistemic enclaves.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from cdlib import algorithms, evaluation, viz, NodeClustering
from IPython.display import Image, display


In [ ]:
def build_demo_graph(seed=42):
    rng = np.random.default_rng(seed)
    communities = [
        {"label": "left-mainstream", "camp": "left", "size": 36, "mean_opinion": -0.55, "enclave": 0},
        {"label": "left-enclave", "camp": "left", "size": 18, "mean_opinion": -0.88, "enclave": 1},
        {"label": "right-mainstream", "camp": "right", "size": 36, "mean_opinion": 0.55, "enclave": 0},
        {"label": "right-enclave", "camp": "right", "size": 18, "mean_opinion": 0.88, "enclave": 1},
    ]
    sizes = [community["size"] for community in communities]
    probability_matrix = [
        [0.18, 0.08, 0.02, 0.004],
        [0.08, 0.25, 0.006, 0.001],
        [0.02, 0.006, 0.18, 0.08],
        [0.004, 0.001, 0.08, 0.25],
    ]

    graph = nx.stochastic_block_model(sizes, probability_matrix, seed=seed)
    graph.graph.clear()

    node_id = 0
    for community in communities:
        for _ in range(community["size"]):
            graph.nodes[node_id]["label"] = f"user_{node_id:03d}"
            graph.nodes[node_id]["community_label"] = community["label"]
            graph.nodes[node_id]["camp"] = community["camp"]
            graph.nodes[node_id]["opinion"] = float(np.clip(rng.normal(community["mean_opinion"], 0.09), -1.0, 1.0))
            graph.nodes[node_id]["enclave"] = int(community["enclave"])
            graph.nodes[node_id]["activity"] = int(rng.poisson(9 if community["enclave"] else 6) + 1)
            node_id += 1

    return graph


In [ ]:
G = build_demo_graph()
print(G)

## 1. Run several community-detection algorithms


In [ ]:
louvain = algorithms.louvain(G)
infomap = algorithms.infomap(G)
label_prop = algorithms.label_propagation(G)

clusterings = [louvain, infomap, label_prop]

pd.DataFrame(
    [{"algorithm": clustering.method_name,
      "communities": len(clustering.communities)} for clustering in clusterings]
)


In [ ]:
comparison_rows = []
reference = louvain
for clustering in clusterings:
    ari = evaluation.adjusted_rand_index(reference, clustering).score
    nmi = evaluation.normalized_mutual_information(reference, clustering).score
    comparison_rows.append(
        {
            "algorithm": clustering.method_name,
            "ARI_vs_louvain": round(ari, 4),
            "NMI_vs_louvain": round(nmi, 4),
        }
    )

pd.DataFrame(comparison_rows)


In [ ]:
cluster_grid = viz.plot_sim_matrix(clusterings, evaluation.adjusted_rand_index)
matrix_path = "module2_similarity_matrix.png"
cluster_grid.fig.savefig(matrix_path, bbox_inches="tight")
display(Image(filename=str(matrix_path)))
plt.close("all")


## 2. Visualize the resulting partitions


In [ ]:
position = nx.spring_layout(G, seed=42)
viz.plot_network_clusters(G, louvain, position=position, figsize=(8, 8), plot_labels=False)
cluster_plot_path = "module2_louvain_network.png"
plt.savefig(cluster_plot_path, bbox_inches="tight")
display(Image(filename=str(cluster_plot_path)))
plt.close("all")


## 3. Evaluate enclave-like structure with CDlib fitness scores


In [ ]:
!wget -O politics-t0.gexf https://github.com/andreafailla/andreafailla.github.io/raw/refs/heads/main/static/uploads/data/politics-t0.gexf

In [ ]:
G = nx.read_gexf('politics-t0.gexf')
print(G)

In [ ]:
# show label distribution
Counter([G.nodes[node]['discrete_leaning'] for node in G.nodes])

In [ ]:
# if not continuous leaning, remove
to_remove = []
for node in G.nodes:
    if not G.nodes[node].get('continuous_leaning'):
        to_remove.append(node)
G.remove_nodes_from(to_remove)

print(G)

In [ ]:
# plot  kde leaning vs. average neighbor leaning
leanings =  [G.nodes[node].get('continuous_leaning', 0) for node in G.nodes]
neighbor_leanings = [np.mean([G.nodes[neighbor].get('continuous_leaning', 0)  for neighbor in G.neighbors(node)]) for node in G.nodes]
plt.figure(figsize=(10, 8))
# seaborn kde
import seaborn as sns
sns.kdeplot(x=leanings, y=neighbor_leanings,
            fill=True, cmap='inferno',
            levels=100, thresh=0,
            )


plt.xlabel('Leaning')
plt.ylabel('Average Neighbor Leaning')
plt.ylim(-1.1, 1.1)
plt.xlim(-1.1, 1.1)
plt.title('Leaning vs. Average Neighbor Leaning')

In [ ]:
louvain = algorithms.louvain(G)
infomap = algorithms.infomap(G)
label_prop = algorithms.label_propagation(G)

clusterings = [louvain, infomap, label_prop]

pd.DataFrame(
    [{"algorithm": clustering.method_name,
      "communities": len(clustering.communities)} for clustering in clusterings]
)


In [ ]:
fitness_rows = []
for clustering in clusterings:
    fitness_rows.append(
        {
            "algorithm": clustering.method_name,
            "modularity": round(evaluation.newman_girvan_modularity(G, clustering).score, 4),
            "internal_edge_density": round(evaluation.internal_edge_density(G, clustering).score, 4),
            "conductance": round(evaluation.conductance(G, clustering).score, 4),
            "hub_dominance": round(evaluation.hub_dominance(G, clustering).score, 4),
        }
    )

fitness_frame = pd.DataFrame(fitness_rows)
fitness_frame


In [ ]:
# viz community graph
viz.plot_community_graph(G, louvain, figsize=(8, 8), top_k=8)

In [ ]:
def purity(nodes):
    labels = [G.nodes[n]["discrete_leaning"] for n in nodes if "discrete_leaning" in G.nodes[n]]
    return Counter(labels).most_common(1)[0][1] / len(labels) if labels else 0


rows = []

for clustering in clusterings:
    for i, community in enumerate(clustering.communities):
        community = [n for n in community if n in G]
        if not community:
            continue

        rows.append({
            "algorithm": clustering.method_name,
            "community_id": f"{clustering.method_name}_{i}",
            "conductance": evaluation.conductance(
                G, NodeClustering([community], G, clustering.method_name)
            ).score,
            "purity": purity(community),
            "size": len(community)
        })

metrics_df = pd.DataFrame(rows)

In [ ]:

plt.figure(figsize=(10, 8))
sns.kdeplot(x=metrics_df['conductance'], y=metrics_df['purity'],
            fill=True, cmap='inferno',
            levels=100, thresh=0.0001,

            )
plt.xlabel('Conductance')
plt.ylabel('Label Purity')
plt.title('Conductance vs. Label Purity')
plt.grid(None)

kde_plot_path = "module2_conductance_purity_kde.png"
plt.savefig(kde_plot_path, bbox_inches="tight")
display(Image(filename=str(kde_plot_path)))
plt.close('all')

In [ ]:
conductance_treshold = 0.5
purity_treshold = 0.5
filter = metrics_df[(metrics_df['conductance'] < conductance_treshold) &
 (metrics_df['purity'] > purity_treshold)]
print(f"Number of epistemic enclaves (cond.<{conductance_treshold}, pur.>{purity_treshold}: {len(filter)}")
print('Users affected:', filter['size'].sum())